# Phase 3C.02 — Predictive Analysis

## Research Objective

Evaluate the 12-month TSMOM signal's predictive ability on Train and Validation using directional classification metrics: Accuracy, Balanced Accuracy, Recall, confusion matrices and naive baselines. Refactor the cleaning logic from 3C.01 into a reusable function before evaluation.

**Scope:** Train (2010–2017) and Validation (2018–2021); Test (2022–2025) excluded.

## 1. Environment and Data Preparation

Reconstruct the predictive dataset using the reusable functions developed in 3C.01: configure imports, rebuild the 16-ETF universe, download monthly prices, calculate 12-month momentum, generate directional signals and targets, split Train and Validation, and clean observations with `clean_signals_and_targets()`. Obtain clean, aligned and reproducible samples before evaluation.

### 1.1. Environment Configuration

Configure the project path and import reusable functions from `src.data` and `src.signals`. Reload modules to use recent changes without restarting the kernel.

In [17]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.signals

importlib.reload(src.data)
importlib.reload(src.signals)

from src.data import download_prices, to_monthly_prices
from src.signals import calculate_momentum, generate_signals, clean_signals_and_targets

### 1.2. Investment Universe and Data Acquisition

Reconstruct the 16-ETF universe across Equities, Commodities, Bonds and Forex. Download adjusted prices for 2008–2021, including 2008–2009 warm-up history. Exclude data from 2022 onward to protect the Test sample.

In [18]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

print(tickers)
print(len(tickers))

prices = download_prices(tickers, "2008-01-01", "2022-01-01", auto_adjust=True, progress=False)
print(prices.shape)
print(prices.index.min())
print(prices.index.max())

['SPY', 'VGK', 'EWJ', 'EEM', 'USO', 'DBA', 'DBB', 'DBC', 'IEF', 'TLT', 'LQD', 'HYG', 'FXE', 'FXY', 'FXB', 'FXA']
16
(3526, 16)
2008-01-02 00:00:00
2021-12-31 00:00:00


### 1.3. Monthly Price Preparation and Validation

Resample daily prices to the last available price of each month. Check dimensions, coverage and missing values before constructing signals.

In [19]:
monthly_prices = to_monthly_prices(prices)
print(monthly_prices.shape)
print(monthly_prices.index.min())
print(monthly_prices.index.max())
print(monthly_prices.isna().sum().sum())

(168, 16)
2008-01-31 00:00:00
2021-12-31 00:00:00
0


### 1.4. Momentum and Directional Signal Construction

Calculate historical 12-month momentum according to the primary specification.

$$
M_{i,t}^{(L)}=\frac{P_{i,t}}{P_{i,t-L}}-1
$$

$$
s_{i,t}^{(L)}=\operatorname{sign}(M_{i,t}^{(L)})
$$

Signals take values +1 (positive), −1 (negative) or 0 (neutral). The first 12 months are undefined due to insufficient history.

**Notation and implementation.** $P_{i,t}$ is ETF $i$'s adjusted month-end price, $t$ is the formation month, and $L=12$ is the lookback in months. $M_{i,t}^{(12)}$ is stored in `momentum_12m`; $s_{i,t}^{(12)}$ is stored in `signals_12m`.

In [20]:

momentum_12m = calculate_momentum(monthly_prices, lookback=12)

print(momentum_12m.head(15))
print(momentum_12m.shape)
print(momentum_12m.isna().sum().sum())

signals_12m = generate_signals(momentum_12m)
signals_12m

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29       NaN       NaN       NaN       NaN       NaN       NaN   
2008-03-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-04-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-05-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-06-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-07-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-08-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-09-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-10-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-11-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-12-31       NaN       NaN       N

Ticker,DBA,DBB,DBC,EEM,EWJ,FXA,FXB,FXE,FXY,HYG,IEF,LQD,SPY,TLT,USO,VGK
Date,,,,,,,,,,,,,,,,
2008-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-02-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-31,1.0,1.0,1.0,1.0,1.0,-1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0
2021-09-30,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0
2021-10-31,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0


### 1.5. Forward Returns and Targets

Forward returns describe the following month's realized return.

$$
r_{i,t+1}=\frac{P_{i,t+1}}{P_{i,t}}-1,\qquad y_{i,t}=\operatorname{sign}(r_{i,t+1})
$$

Targets take values +1, −1 or 0. Both signals and targets are indexed by the formation date $t$; forward returns are not used to construct signals.

**Outputs:** `forward_returns_1m` stores $r_{i,t+1}$ at date $t$; `targets_1m` stores $y_{i,t}$, the sign of that realized return. $P_{i,t}$ denotes the adjusted month-end price of ETF $i$.

In [21]:
forward_returns_1m = monthly_prices.pct_change().shift(-1)
print(forward_returns_1m.head(3))
print(forward_returns_1m.tail(3))

targets_1m = generate_signals(forward_returns_1m)
print(targets_1m.head(3))
print(targets_1m.tail(3))

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31  0.123547  0.109278  0.111453  0.019943 -0.014937  0.044435   
2008-02-29 -0.122955 -0.062825 -0.009141 -0.037531 -0.012770 -0.015464   
2008-03-31  0.004115  0.000000  0.060386  0.091383  0.073565  0.037449   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2008-01-31  0.001964  0.023234  0.023640 -0.018694  0.012393 -0.007063   
2008-02-29  0.002768  0.042211  0.042755  0.009461  0.013373 -0.005972   
2008-03-31  0.005463 -0.008206 -0.042797  0.033969 -0.024173  0.012447   

Ticker           SPY       TLT       USO       VGK  
Date                                                
2008-01-31 -0.025843 -0.004553  0.111541 -0.002937  
2008-02-29 -0.008942  0.021389  0.011689  0.013107  
2008-03-31  0.047662 -0.024864  0.136922  0.04

### 1.6. Chronological Train / Validation Split

- Train: realized returns January 2010–December 2017.
- Validation: realized returns January 2018–December 2021.
- Test: realized returns January 2022–December 2025, excluded.

Because a signal formed at month-end $t$ predicts month $t+1$, formation dates are selected one month earlier than realized return dates.

In [22]:
train_signals = signals_12m.loc["2009-12-31":"2017-11-30"]
train_targets = targets_1m.loc["2009-12-31":"2017-11-30"]

print(train_signals.shape)
print(train_signals.notna().sum().sum())
print(train_targets.shape)
print(train_targets.notna().sum().sum())

print("-" * 150)

validation_signals = signals_12m.loc["2017-12-31":"2021-11-30"]
validation_targets = targets_1m.loc["2017-12-31":"2021-11-30"]

print(validation_signals.shape)
print(validation_signals.notna().sum().sum())
print(validation_targets.shape)
print(validation_targets.notna().sum().sum())

(96, 16)
1536
(96, 16)
1536
------------------------------------------------------------------------------------------------------------------------------------------------------
(48, 16)
768
(48, 16)
768


### 1.7. Observation Cleaning and Validation

Use `clean_signals_and_targets()` to identify nonmissing, nonzero signal-target pairs for binary classification.

$$
m_{i,t}=\mathbb{1}\{s_{i,t}\in\{-1,+1\},\ y_{i,t}\in\{-1,+1\}\},\qquad N_{\mathrm{valid}}=\sum_i\sum_t m_{i,t}
$$

Preserve DataFrame dimensions and temporal alignment. Verify valid observation counts separately for Train and Validation.

**Notation and implementation.** $\mathbb{1}$ is the indicator function; $m_{i,t}$ is `train_mask` or `validation_mask`, and $N_{\mathrm{valid}}$ is the total number of true entries in the corresponding mask.

In [23]:
clean_train_signals, clean_train_targets, train_mask = clean_signals_and_targets(train_signals, train_targets)

clean_validation_signals, clean_validation_targets, validation_mask = clean_signals_and_targets(validation_signals, validation_targets)

print(clean_train_signals.shape)
print(clean_train_signals.notna().sum().sum())
print(clean_train_signals.isna().sum().sum())

print("-" * 10)

print(clean_train_targets.shape)
print(clean_train_targets.notna().sum().sum())
print(clean_train_targets.isna().sum().sum())

print("-" * 50)

print(clean_validation_signals.shape)
print(clean_validation_signals.notna().sum().sum())
print(clean_validation_signals.isna().sum().sum())

print("-" * 10)

print(clean_validation_targets.shape)
print(clean_validation_targets.notna().sum().sum())
print(clean_validation_targets.isna().sum().sum())

print("-" * 50) 

print(clean_train_signals.notna().equals(clean_train_targets.notna()))
print(clean_validation_signals.notna().equals(clean_validation_targets.notna()))

(96, 16)
1531
5
----------
(96, 16)
1531
5
--------------------------------------------------
(48, 16)
765
3
----------
(48, 16)
765
3
--------------------------------------------------
True
True


### Conclusion — Data Preparation

- Universe: 16 ETFs in four asset classes.
- Frequency: 168 monthly observations, January 2008–December 2021.
- Primary signal: direction of historical 12-month momentum.
- Target: next-month return direction.
- Train: 1,531 valid observations out of 1,536.
- Validation: 765 valid observations out of 768.

Signal-target alignment and common validity masks were verified. Neutral and missing observations are excluded without changing DataFrame structure. Test remains excluded; samples are ready for predictive evaluation.

## 2. Predictive Performance Evaluation

Evaluate the ability of 12-month TSMOM to predict next-month return directions using binary classification metrics on Train and Validation separately.

### 2.1. Accuracy

Accuracy is the proportion of correct directional predictions among valid observations.

$$
A=\frac{TP+TN}{TP+TN+FP+FN}
$$

Missing and neutral observations are excluded using the validity masks.

**Notation and implementation.** $TP$ and $TN$ count correct positive and negative predictions; $FP$ and $FN$ count incorrect positive and negative predictions, respectively. Their sum is the valid observation count. $A$ is `train_accuracy` or `validation_accuracy`, calculated directly as the number of correct predictions divided by the mask count.

In [24]:
train_correct_predictions = clean_train_signals == clean_train_targets
print(train_correct_predictions.sum())
print(train_correct_predictions.sum().sum())

train_accuracy = (train_correct_predictions.sum().sum()) / (train_mask.sum().sum())
print(f"Train accuracy : {train_accuracy:.2%}")

print("-" * 150)

validation_correct_predictions = clean_validation_signals == clean_validation_targets
print(validation_correct_predictions.sum())
print(validation_correct_predictions.sum().sum())

validation_accuracy = (validation_correct_predictions.sum().sum()) / (validation_mask.sum().sum())
print(f"Validation accuracy : {validation_accuracy:.2%}")

Ticker
DBA    46
DBB    45
DBC    50
EEM    49
EWJ    50
FXA    44
FXB    38
FXE    45
FXY    57
HYG    65
IEF    45
LQD    56
SPY    65
TLT    40
USO    45
VGK    47
dtype: int64
787
Train accuracy : 51.40%
------------------------------------------------------------------------------------------------------------------------------------------------------
Ticker
DBA    30
DBB    22
DBC    26
EEM    23
EWJ    20
FXA    20
FXB    20
FXE    29
FXY    24
HYG    26
IEF    25
LQD    28
SPY    31
TLT    21
USO    25
VGK    21
dtype: int64
391
Validation accuracy : 51.11%


**Results — Accuracy**

12-month TSMOM achieves 51.40% Accuracy on Train (787/1,531) and 51.11% on Validation (391/765). The 0.29 percentage-point difference suggests descriptive stability, but Accuracy slightly above 50% does not establish predictive value; class distributions and baselines must be considered.

### 2.2. Balanced Accuracy

Accuracy can be misleading when positive and negative targets are imbalanced. Balanced Accuracy averages the two class-specific Recalls.

$$
\mathrm{Recall}_+=\frac{TP}{TP+FN},\qquad \mathrm{Recall}_-=\frac{TN}{TN+FP},\qquad BA=\frac{\mathrm{Recall}_++\mathrm{Recall}_-}{2}
$$

Analyze class distributions and calculate Balanced Accuracy on Train and Validation to determine whether the signal identifies both directions rather than benefiting mainly from one class's prevalence.

**Outputs:** $\mathrm{Recall}_+$ is the fraction of actual positive returns correctly predicted; $\mathrm{Recall}_-$ is the fraction of actual negative returns correctly predicted. For Train, these are `positive_train_recall` and `negative_train_recall`; $BA$ is `train_balanced_accuracy`. The equivalent Validation variables use the same definitions. The denominators are counts of realized positive and negative targets.

In [25]:
positive_train_targets = (clean_train_targets == 1).sum().sum()
negative_train_targets = (clean_train_targets == -1).sum().sum()

print(positive_train_targets)
print(negative_train_targets)
print((positive_train_targets + negative_train_targets) == clean_train_targets.notna().sum().sum())

print("-" * 50)

positive_correct_train_predictions = (clean_train_signals == 1) & (clean_train_targets == 1)
print(positive_correct_train_predictions.sum().sum())

negative_correct_train_predictions = (clean_train_signals == -1) & (clean_train_targets == -1)
print(negative_correct_train_predictions.sum().sum())

positive_train_recall = (positive_correct_train_predictions.sum().sum() / positive_train_targets)
negative_train_recall = (negative_correct_train_predictions.sum().sum() / negative_train_targets)

print(f"Positive train recall : {positive_train_recall:.2%}")
print(f"Negative train recall : {negative_train_recall:.2%}")

print("-" * 50)

train_balanced_accuracy = (positive_train_recall + negative_train_recall) / 2

print(f" Train balanced accuracy : {train_balanced_accuracy:.2%}")

print("-" * 150)

positive_validation_targets = (clean_validation_targets == 1).sum().sum()
negative_validation_targets = (clean_validation_targets == -1).sum().sum()

print(positive_validation_targets)
print(negative_validation_targets)
print((positive_validation_targets + negative_validation_targets) == clean_validation_targets.notna().sum().sum())

print("-" * 50)

positive_correct_validation_predictions = (clean_validation_signals == 1) & (clean_validation_targets == 1)
print(positive_correct_validation_predictions.sum().sum())

negative_correct_validation_predictions = (clean_validation_signals == -1) & (clean_validation_targets == -1)
print(negative_correct_validation_predictions.sum().sum())

positive_validation_recall = (positive_correct_validation_predictions.sum().sum() / positive_validation_targets)
negative_validation_recall = (negative_correct_validation_predictions.sum().sum() / negative_validation_targets)

print(f"Positive validation recall : {positive_validation_recall:.2%}")
print(f"Negative validation recall : {negative_validation_recall:.2%}")

print("-" * 50)

validation_balanced_accuracy = (positive_validation_recall + negative_validation_recall) / 2

print(f"Validation balanced accuracy : {validation_balanced_accuracy:.2%}")

824
707
True
--------------------------------------------------
494
293
Positive train recall : 59.95%
Negative train recall : 41.44%
--------------------------------------------------
 Train balanced accuracy : 50.70%
------------------------------------------------------------------------------------------------------------------------------------------------------
420
345
True
--------------------------------------------------
249
142
Positive validation recall : 59.29%
Negative validation recall : 41.16%
--------------------------------------------------
Validation balanced accuracy : 50.22%


**Results — Balanced Accuracy**

- Train: 50.70%.
- Validation: 50.22%.

Recall is persistently asymmetric: approximately 59% of positive returns and 41% of negative returns are correctly identified. Despite relative stability, Balanced Accuracy remains near 50%, so significant predictive ability has not been established. Classification errors and baselines require further examination.

### 2.3. Confusion Matrix

Analyze directional classification errors on both samples:

- TP: positive return correctly predicted.
- TN: negative return correctly predicted.
- FP: positive prediction, negative realization.
- FN: negative prediction, positive realization.

With predicted classes in rows and realized classes in columns, in positive/negative order:

$$
C=\begin{pmatrix}TP&FP\\FN&TN\end{pmatrix},\qquad TP+TN+FP+FN=N_{\mathrm{valid}}
$$

The matrices complement Accuracy and Balanced Accuracy by revealing the structure of correct and incorrect predictions.

**Output:** $C$ is the count matrix stored in `train_confusion_matrix` or `validation_confusion_matrix`. $N_{\mathrm{valid}}$ counts all valid signal-target pairs; the four cells must sum to this count.

In [26]:
train_true_positive = positive_correct_train_predictions
train_true_negative = negative_correct_train_predictions
train_false_positive = (clean_train_signals == 1) & (clean_train_targets == -1)
train_false_negative = (clean_train_signals == -1) & (clean_train_targets == 1)

print(f"Train true positive predictions : {train_true_positive.sum().sum()}")
print(f"Train false positive predictions : {train_false_positive.sum().sum()}")
print(f"Sum of train positive predictions : {(train_true_positive.sum().sum()) + (train_false_positive.sum().sum())}")

print ("")

print(f"Train true negative predictions : {train_true_negative.sum().sum()}")
print(f"Train false negative predictions : {train_false_negative.sum().sum()}")
print(f"Sum of train negative predictions : {(train_true_negative.sum().sum()) + (train_false_negative.sum().sum())}")

print(((train_true_positive.sum().sum()) + (train_false_positive.sum().sum()) + (train_true_negative.sum().sum()) + (train_false_negative.sum().sum())) == clean_train_targets.notna().sum().sum())

print("")
print("-" * 50)
print("")

validation_true_positive = positive_correct_validation_predictions
validation_true_negative = negative_correct_validation_predictions
validation_false_positive = (clean_validation_signals == 1) & (clean_validation_targets == -1)
validation_false_negative = (clean_validation_signals == -1) & (clean_validation_targets == 1)

print(f"Validation true positive predictions : {validation_true_positive.sum().sum()}")
print(f"Validation false positive predictions : {validation_false_positive.sum().sum()}")
print(f"Sum of validation positive predictions : {(validation_true_positive.sum().sum()) + (validation_false_positive.sum().sum())}")

print("")

print(f"Validation true negative predictions : {validation_true_negative.sum().sum()}")
print(f"Validation false negative predictions : {validation_false_negative.sum().sum()}")
print(f"Sum of validation negative predictions : {(validation_true_negative.sum().sum()) + (validation_false_negative.sum().sum())}")

print(((validation_true_positive.sum().sum()) + (validation_false_positive.sum().sum()) + (validation_true_negative.sum().sum()) + (validation_false_negative.sum().sum())) == clean_validation_targets.notna().sum().sum())

Train true positive predictions : 494
Train false positive predictions : 414
Sum of train positive predictions : 908

Train true negative predictions : 293
Train false negative predictions : 330
Sum of train negative predictions : 623
True

--------------------------------------------------

Validation true positive predictions : 249
Validation false positive predictions : 203
Sum of validation positive predictions : 452

Validation true negative predictions : 142
Validation false negative predictions : 171
Sum of validation negative predictions : 313
True


**Confusion Matrices — Train / Validation**

The 2 × 2 matrices use predicted directions in rows and realized directions in columns. TP and TN on the main diagonal are correct predictions; FP and FN are errors. This representation supports consistency checks against previously calculated classification metrics.

In [27]:
import pandas as pd

train_confusion_matrix = pd.DataFrame(
                            data=[[train_true_positive.sum().sum(), train_false_positive.sum().sum()],
                                [train_false_negative.sum().sum(), train_true_negative.sum().sum()]
                                ],
                                index=["Signal +1", "Signal -1"],
                                columns=["Target +1", "Target -1"]    
                        )

validation_confusion_matrix = pd.DataFrame(
                                data=[[validation_true_positive.sum().sum(), validation_false_positive.sum().sum()],
                                    [validation_false_negative.sum().sum(), validation_true_negative.sum().sum()]
                                    ],
                                    index=["Signal +1", "Signal -1"],
                                    columns=["Target +1", "Target -1"]
                            )

display(train_confusion_matrix)
print("")
print("")
display(validation_confusion_matrix)

print("")
print("")

display(train_confusion_matrix.sum(axis=0))
display(validation_confusion_matrix.sum(axis=0))

,Target +1,Target -1
Signal +1,494,414
Signal -1,330,293


,Target +1,Target -1
Signal +1,249,203
Signal -1,171,142


Target +1    824
Target -1    707
dtype: int64

Target +1    420
Target -1    345
dtype: int64

**Classification Summary — Train / Validation**

The table combines TP, FP, TN and FN for Train and Validation to compare correct predictions and errors. Counts are calculated directly from valid observations rather than entered manually.

In [28]:
confusion_summary = {"Train" :
                          {"True positive predictions" : train_true_positive.sum().sum(),
                           "False positive predictions" : train_false_positive.sum().sum(),
                           "True negative predictions" : train_true_negative.sum().sum(),
                           "False negative predictions" : train_false_negative.sum().sum()},
                    "Validation" :
                           {"True positive predictions" : validation_true_positive.sum().sum(),
                            "False positive predictions" : validation_false_positive.sum().sum(),
                            "True negative predictions" : validation_true_negative.sum().sum(),
                            "False negative predictions" : validation_false_negative.sum().sum()}
                          }

confusion_summary_df = pd.DataFrame(confusion_summary).T
confusion_summary_df

,True positive predictions,False positive predictions,True negative predictions,False negative predictions
Train,494,414,293,330
Validation,249,203,142,171


**Results and Interpretation — Confusion Matrix**

Train contains 787 correct and 744 incorrect predictions; Validation contains 391 correct and 374 incorrect predictions.

- Train: 908 positive predictions versus 623 negative predictions.
- Validation: 452 positive predictions versus 313 negative predictions.

This asymmetry is consistent with the higher Positive Recall observed earlier. Error structure is relatively similar across periods, but classification performance remains limited and exhibits an upward directional bias. Baseline comparisons are required to assess added predictive value.

### 2.4. Baseline Comparisons

Compare TSMOM against constant directional classifiers on the same valid observations in each period. Always Long predicts +1; Always Short predicts −1.

$$
A^{AL}=\frac{TP+FN}{N_{\mathrm{valid}}},\qquad A^{AS}=\frac{TN+FP}{N_{\mathrm{valid}}}
$$

When both classes occur, $BA^{AL}=BA^{AS}=0.5$.

Accuracy above 50% does not necessarily establish an advantage when realized return directions are imbalanced.

**Notation and implementation.** $TP+FN$ counts positive realized targets and $TN+FP$ negative realized targets, using the strategy confusion counts. $N_{\mathrm{valid}}$ is the total valid count. $A^{AL}$ and $A^{AS}$ are the Always Long and Always Short accuracies (`train_always_long_accuracy`, `train_always_short_accuracy`, and their Validation equivalents).

In [29]:
train_always_long_accuracy = (positive_train_targets / train_mask.sum().sum())
print(f"Train always long accuracy : {train_always_long_accuracy:.2%}")

train_always_short_accuracy = (negative_train_targets / train_mask.sum().sum())
print(f"Train always negative accuracy : {train_always_short_accuracy:.2%}")

train_always_long_balanced_accuracy = (1 + 0) / 2 
print(f"Train always long balanced accuracy : {train_always_long_balanced_accuracy:.2%}")

print("-" * 50)

validation_always_long_accuracy = (positive_validation_targets / validation_mask.sum().sum())
print(f"Validation always long accuracy : {validation_always_long_accuracy:.2%}")

validation_always_short_accuracy = (negative_validation_targets / validation_mask.sum().sum())
print(f"Validation always short accuracy : {validation_always_short_accuracy:.2%}")

validation_always_long_balanced_accuracy = (1 + 0) / 2
print(f"Validation always long balanced accuracy : {validation_always_long_balanced_accuracy:.2%}")

Train always long accuracy : 53.82%
Train always negative accuracy : 46.18%
Train always long balanced accuracy : 50.00%
--------------------------------------------------
Validation always long accuracy : 54.90%
Validation always short accuracy : 45.10%
Validation always long balanced accuracy : 50.00%


**Results and Interpretation — Baseline Comparisons**

- Train: momentum Accuracy 51.40% versus Always Long 53.82%.
- Validation: momentum Accuracy 51.11% versus Always Long 54.90%.
- Both baselines have Balanced Accuracy of 50%.
- Momentum Balanced Accuracy is 50.70% on Train and 50.22% on Validation.

Momentum underperforms Always Long in Accuracy. Its slight Balanced Accuracy advantage does not establish statistical significance. Descriptive evidence is insufficient to establish robust predictive value; stability analysis and suitable statistical tests are required.

## 3. Train / Validation Comparison

Consolidate classification metrics, measure performance differences, assess class-specific Recall stability and compare against directional baselines. For a metric $Q$ expressed as a decimal:

$$
\Delta Q_{\mathrm{pp}}=100(Q_{\mathrm{Validation}}-Q_{\mathrm{Train}})
$$

Descriptive stability does not establish statistically significant predictive ability.

**Output:** $\Delta Q_{\mathrm{pp}}$ measures the Validation-minus-Train change in percentage points; it is the `Difference (pp)` column of `performance_summary_df`. For example, a decimal change of 0.02 equals +2 pp.

In [ ]:
performance_summary = {"Train" :
                       {"Accuracy" : train_accuracy,
                        "Balanced Accuracy" : train_balanced_accuracy,
                        "Positive Recall" : positive_train_recall,
                        "Negative Recall" : negative_train_recall,
                        "Always Long Accuracy" : train_always_long_accuracy,
                        "Always Short Accuracy" : train_always_short_accuracy},
                        "Validation":
                        {"Accuracy" : validation_accuracy,
                         "Balanced Accuracy" : validation_balanced_accuracy,
                         "Positive Recall" : positive_validation_recall, 
                         "Negative Recall" : negative_validation_recall,
                         "Always Long Accuracy" : validation_always_long_accuracy,
                         "Always Short Accuracy" : validation_always_short_accuracy}
                        }

performance_summary_df = pd.DataFrame(performance_summary)
# display(performance_summary_df.style.format("{:.2%}"))

performance_summary_df["Difference (pp)"] = (performance_summary_df["Validation"] - performance_summary_df["Train"]) * 100
display(performance_summary_df.style.format({"Train" : "{:.2%}",
                                             "Validation" : "{:.2%}",
                                             "Difference (pp)" : "{:+.4f}"
                                             }))

,Train,Validation,Difference (pp)
Accuracy,51.40%,51.11%,-0.2932
Balanced Accuracy,50.70%,50.22%,-0.4745
Positive Recall,59.95%,59.29%,-0.6657
Negative Recall,41.44%,41.16%,-0.2833
Always Long Accuracy,53.82%,54.90%,+1.0809
Always Short Accuracy,46.18%,45.10%,-1.0809


: 

### Conclusion — Train / Validation Comparison

Accuracy decreases from 51.40% to 51.11% (−0.29 pp), and Balanced Accuracy from 50.70% to 50.22% (approximately −0.47 pp using unrounded values). Positive and negative Recalls remain relatively stable, with better detection of positive returns. Always Long exceeds momentum Accuracy in both samples, partly because positive returns are more frequent.

The signal's direction classification is relatively stable but weak, with Balanced Accuracy close to 50%. No convincing predictive advantage is established. The analysis remains descriptive and does not account for statistical uncertainty, temporal or cross-asset dependence, or economic strategy performance. Further analysis by asset class, lookback and period is necessary.

## 4. General Conclusion — Predictive Analysis

### Research Objective

Evaluate the directional predictive power of 12-month TSMOM across 16 ETFs in four asset classes using Train and Validation only; Test remains excluded.

### Key Results

| Metric | Train | Validation |
|---|---:|---:|
| Valid observations | 1,531 | 765 |
| Accuracy | 51.40% | 51.11% |
| Balanced Accuracy | 50.70% | 50.22% |
| Positive Recall | 59.95% | 59.29% |
| Negative Recall | 41.44% | 41.16% |
| Always Long Accuracy | 53.82% | 54.90% |

### Main Findings

1. **Limited predictive performance:** Balanced Accuracy remains close to 50% in both periods despite Accuracy slightly exceeding 50%.
2. **Directional asymmetry:** approximately 59% of positive and 41% of negative returns are correctly identified, persistently across samples.
3. **Descriptive stability:** small differences across periods do not establish robustness when performance is weak.
4. **Baseline underperformance:** Always Long exceeds TSMOM Accuracy in both periods, illustrating the importance of class imbalance.

### Methodological Limitations

No significance tests have yet been conducted. Monthly observations and ETFs may be dependent across time and assets. Aggregate metrics may conceal asset or class differences. Predictive metrics do not establish economic profitability, transaction costs or strategy risk. Further robustness testing is required.

### Final Assessment

The 12-month signal does not currently provide convincing evidence of directional predictive ability in this universe. These results neither establish nor definitively reject TSMOM effectiveness; they require robustness analysis and appropriate statistical validation.

### Next Research Step

Phase 3C.03 will examine cross-asset heterogeneity, robustness and predefined lookbacks of 3, 6 and 12 months. The 12-month reference specification remains fixed rather than selecting the historically strongest alternative. Test remains reserved for Phase 3F.